## PRACTICA 7


Contexto del Problema
El gobierno estatal desea diseñar una política pública orientada a mejorar el bienestar de los hogares, particularmente aquellos con mayor vulnerabilidad económica. Para ello, necesita comprender qué factores socioeconómicos determinan el consumo mensual de los hogares, dado que el consumo es un buen proxy del nivel de vida y del dinamismo económico.


datos_hogar.csv
 

Se cuenta con una base de datos de hogares que incluye información demográfica, económica y estructural:

ingreso: ingreso total del hogar.

edad_jefe: edad del jefe de hogar.

tamano_hogar: número total de miembros del hogar.

numero_dependientes: número de dependientes en el hogar.

educacion: nivel educativo del jefe de hogar (categórica).

area_urbana: 1 si el hogar está en zona urbana, 0 si rural.

propietario: 1 si el hogar es propietario de su vivienda, 0 si no.

consumo_mensual: gasto mensual total del hogar (variable objetivo).

In [3]:
#### Importamos las paqueterias necesarias
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from itertools import combinations
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix
#from ydata_profiling import ProfileReport


In [4]:
#Question 1
#Sube la base de datos, explora su información y sus descriptivos, y realiza un reporte automatizado.
#Según el reporte, la base de datos tiene 3 variables categóricas. ¿Cuáles son?

# Cargar la base de datos
df = pd.read_csv("datos_hogar.csv")

# Exploración básica
print(df.head())
print(df.info())
print(df.describe())

# Reporte automatizado
#profile = ProfileReport(df, title="Datos Hogar Report")
#profile.to_file("datos_hogar_report.html")

        ingreso  edad_jefe  tamano_hogar  numero_dependientes     educacion  \
0  23973.713224         59             7                    0         prepa   
1  18893.885591         20             2                    0         prepa   
2  25181.508305         75             3                    1      primaria   
3  32184.238851         73             3                    0  licenciatura   
4  18126.773002         23             3                    0         prepa   

   area_urbana  propietario  consumo_mensual  
0            0            1     18431.018621  
1            1            1     13130.202707  
2            1            0     12868.509824  
3            0            1     22423.800560  
4            0            0      9335.810752  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ingreso              5000 non-nu

#### RESPUESTAS 
- education
- area_urban
- propiterio

##### 2. Con base en el reporte responde: ¿cuál de las variables presenta outliers?
- ingreso
- consumo_mensual


##### 3 Según el reporte ¿la variable area_urbana presenta algún tipo de sesgo? en caso de ser afirmativo ¿cuál sería el sesgo?
- Si, sesgo de representación


##### Question 4 ¿Cuál es la variable de interés (la variable a predecir), según el contexto?
- consumo_mensual

##### 5 Con base en el reporte, ordena las variables de MAYOR correlación a MENOR correlación que tengan con la variable a predecir, es decir, la variable objetivo.

In [ ]:
# Correlaciones con consumo_mensual ordenadas de mayor a menor (valor absoluto)
correlaciones = df.corr()['consumo_mensual'].drop('consumo_mensual')
correlaciones_ordenadas = correlaciones.abs().sort_values(ascending=False)
print(correlaciones_ordenadas)

ValueError: could not convert string to float: 'prepa'

##### RESPUESTAS 5 
1. ingreso
2. tamano_hogar
3. numero_dependientes
4. propietario
5. area_urbana
6. educacion
7. edad_jefe

##### 6 Según el reporte ¿cuál es el 3er nivel educativo más frecuente en la muestra?
- licenciatura

Question 7

Limpia las cosas que puedas limpiar antes de hacer el split de la base. Cuando estés transformando la variable educacion, asegúrate de que tome los valores de la siguiente forma:

‘primaria’: 0, ‘secundaria’: 1, ‘prepa’: 2, ‘licenciatura’: 3, ‘posgrado’: 4



Una vez que hiciste la limpieza a priori del split, ahora sí haz el split con un tamaño de conjunto de entrenamiento del 70% de los datos y con un random_state=42.

Después de todo lo anterior, para asegurar que hiciste bien todo el proceso, ¿cuál es el promedio de consumo_mensual en el conjunto de entrenamiento?

In [10]:
# 1. Revisar duplicados y eliminarlos
print("Duplicados:", df.duplicated().sum())
df = df.drop_duplicates()

# 2. Revisar nulos
print("Nulos:\n", df.isnull().sum())

# 3. Transformar educacion a ordinal
educacion_map = {'primaria': 0, 'secundaria': 1, 'prepa': 2, 'licenciatura': 3, 'posgrado': 4}
df['educacion'] = df['educacion'].map(educacion_map)

# 4. Split 70/30
X = df.drop(columns='consumo_mensual')
y = df['consumo_mensual']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

# 5. Verificación
train_df = X_train.copy()
train_df['consumo_mensual'] = y_train
print("Promedio consumo_mensual (entrenamiento):", train_df['consumo_mensual'].mean())

Duplicados: 0
Nulos:
 ingreso                0
edad_jefe              0
tamano_hogar           0
numero_dependientes    0
educacion              0
area_urbana            0
propietario            0
consumo_mensual        0
dtype: int64
Promedio consumo_mensual (entrenamiento): 13953.302427513983


Question 8

Ahora que hiciste el split, limpia el conjunto de entrenamiento. A las variables numéricas aplícales winsorización para tratar los outliers; una vez hecho eso, revisa que no se hayan generado duplicados.

Para saber que hiciste de forma correcta el proceso, ¿cuántas observaciones te quedaron en el conjunto de entrenamiento después de la limpieza?

In [11]:
from scipy.stats.mstats import winsorize

# 1. Winsorización a variables numéricas en entrenamiento
# (aplicar solo sobre X_train para no hacer data leakage)

numeric_cols = ['ingreso', 'edad_jefe', 'tamano_hogar', 'numero_dependientes', 'consumo_mensual']

# Unimos X_train y y_train para trabajar juntos
train_df = X_train.copy()
train_df['consumo_mensual'] = y_train

# Aplicar winsorización (límite 1% en cada cola)
for col in numeric_cols:
    train_df[col] = winsorize(train_df[col], limits=[0.01, 0.01])

# 2. Eliminar duplicados que pudieran haberse generado
print("Duplicados antes:", train_df.duplicated().sum())
train_df = train_df.drop_duplicates()
print("Duplicados después:", train_df.duplicated().sum())

# 3. Observaciones finales
print("Observaciones en entrenamiento después de limpieza:", len(train_df))

# 4. Separar de nuevo X_train y y_train limpios
X_train = train_df.drop(columns='consumo_mensual')
y_train = train_df['consumo_mensual']

Duplicados antes: 1
Duplicados después: 0
Observaciones en entrenamiento después de limpieza: 3499


Question 9

Ya que terminaste de limpiar el conjunto de entrenamiento, apliquemos el modelo de RLM a dicho conjunto. Calcula las métricas del RLM sobre el conjunto de entrenamiento y responde: ¿cuál fue el MAE?

In [12]:
# Modelo de Regresión Lineal Múltiple sobre conjunto de entrenamiento

# Agregar constante para el intercepto
X_train_sm = sm.add_constant(X_train)

# Ajustar modelo
modelo = sm.OLS(y_train, X_train_sm).fit()

# Resumen del modelo
print(modelo.summary())

# Predicciones sobre entrenamiento
y_pred_train = modelo.predict(X_train_sm)

# Métricas
mae = mean_absolute_error(y_train, y_pred_train)
mse = mean_squared_error(y_train, y_pred_train)
rmse = np.sqrt(mse)
r2 = r2_score(y_train, y_pred_train)

print(f"MAE:  {mae:.4f}")
print(f"MSE:  {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R2:   {r2:.4f}")

                            OLS Regression Results                            
Dep. Variable:        consumo_mensual   R-squared:                       0.880
Model:                            OLS   Adj. R-squared:                  0.880
Method:                 Least Squares   F-statistic:                     3666.
Date:                Tue, 10 Mar 2026   Prob (F-statistic):               0.00
Time:                        21:29:50   Log-Likelihood:                -31636.
No. Observations:                3499   AIC:                         6.329e+04
Df Residuals:                    3491   BIC:                         6.334e+04
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const               -3485.0934    

#####  Question 10

Ahora apliquemos la técnica de validación cruzada para verificar que el modelo es consistente. Utiliza un k = 5.

¿Cuál es el MAE promedio según la técnica de k-folds?

In [13]:
from sklearn.linear_model import HuberRegressor
from sklearn.model_selection import cross_val_score

modelo_cv = HuberRegressor(max_iter=500)
cv_scores = cross_val_score(modelo_cv, X, y, cv=5, scoring='neg_mean_absolute_error')
mae_scores = -cv_scores

print(f"MAE por fold: {mae_scores}")
print(f"MAE promedio: {mae_scores.mean():.4f}")

MAE por fold: [1720.05490617 1845.40698424 1843.32413589 1724.69485802 1685.8319227 ]
MAE promedio: 1763.8626


Question 11

Ahora aplica exactamente las mismas técnicas de limpieza que hiciste en el conjunto de entrenamiento, pero ahora en el conjunto de prueba.

RECUERDA: debes utilizar en la winsorización los mismos parámetros del conjunto de entrenamiento.

Para saber si realizaste bien la transformación, calcula el consumo_promedio en el conjunto de prueba.

Nota: Utiliza hasta dos decimales después del punto, no importa si redondeas o no.

In [ ]:
# Unimos X_test y y_test para trabajar juntos
test_df = X_test.copy()
test_df['consumo_mensual'] = y_test

# Aplicar winsorización con los MISMOS límites que en entrenamiento
# Los límites de winsorize son fijos (1% cada cola), pero los valores
# se calculan sobre el conjunto de entrenamiento y se aplican al test

# Obtener límites del conjunto de entrenamiento
from scipy.stats import mstats

for col in numeric_cols:
    # Calcular los límites desde train
    lower = np.percentile(train_df[col], 1)
    upper = np.percentile(train_df[col], 99)
    
    # Aplicar esos mismos límites al test (clip)
    test_df[col] = test_df[col].clip(lower=lower, upper=upper)

# Eliminar duplicados
print("Duplicados antes:", test_df.duplicated().sum())
test_df = test_df.drop_duplicates()
print("Duplicados después:", test_df.duplicated().sum())

# Separar de nuevo
X_test = test_df.drop(columns='consumo_mensual')
y_test = test_df['consumo_mensual']

# Consumo promedio en test
print(f"Consumo promedio en prueba: {test_df['consumo_mensual'].mean():.2f}")

Duplicados antes: 0
Duplicados después: 0
Consumo promedio en prueba: 14105.50


Question 12

Ahora vamos a evaluar el desempeño del modelo con el conjunto de prueba. Calcula las métricas y responde: ¿De cuánto es el MAE?

In [ ]:
# Predicciones sobre conjunto de prueba
X_test_sm = sm.add_constant(X_test)

y_pred_test = modelo.predict(X_test_sm)

# Métricas
mae_test = mean_absolute_error(y_test, y_pred_test)
mse_test = mean_squared_error(y_test, y_pred_test)
rmse_test = np.sqrt(mse_test)
r2_test = r2_score(y_test, y_pred_test)

print(f"MAE:  {mae_test:.4f}")
print(f"MSE:  {mse_test:.4f}")
print(f"RMSE: {rmse_test:.4f}")
print(f"R2:   {r2_test:.4f}")

MAE:  1641.7641
MSE:  4190096.8803
RMSE: 2046.9726
R2:   0.8810


Question 13

¿De cuánto es el error relativo aproximadamente?

Nota: No utilices el símbolo %, con dos decimales después del punto es suficiente, no importa el redondeo.

In [ ]:
# Error relativo (MAE / promedio real de y_test)
error_relativo = mae_test / y_test.mean()
print(f"Error relativo: {error_relativo:.4f}")
print(f"Error relativo (%): {error_relativo * 100:.2f}%")

Error relativo: 0.1164
Error relativo (%): 11.64%


##### Question 14

Interpreta los resultados de las métricas evaluadas en el conjunto de prueba. ¿Cuál de las siguientes afirmaciones es cierta?
El error promedio es moderado y razonable respecto al nivel de consumo.
		
No parece haber sobreajuste severo, ya que las métricas provienen del conjunto de entrenamiento.

Si el consumo promedio ronda, por ejemplo, $15,000–$20,000, el error representa aproximadamente entre 10% y 14%, lo cual es razonable para datos microeconómicos.
		
El modelo explica aproximadamente el 88.11% de la variación del consumo mensual de los hogares.
		
Para un modelo lineal, el desempeño es sólido.
		
El modelo tiene baja poder explicativo (R² alto).
		
El modelo acierta, en promedio, en alrededor de $1,641 por hogar.

- ✅ El error promedio es moderado y razonable respecto al nivel de consumo
- ✅ El modelo explica aproximadamente el 88.11% de la variación del consumo mensual
- ✅ Para un modelo lineal, el desempeño es sólido
- ✅ Si el consumo ronda $15,000–$20,000, el error representa entre 10% y 14%